# Python 从入门到实践（第 3 版）· 第 1–2 周通关笔记

**配套教材**：`book/python编程从入门到实践（第三版）.pdf`（中文版 727 页）
**学习节奏**：按《Python 实验数据建模 12 周学习手册》第 1–2 周「Python：够用即可」编排
**这两周的目标**：能读写文件、能循环判断、能定义函数、能用列表和字典。会了这四件事，pandas 和 sklearn 就只是调 API。

## 怎么用（第一次来先看这段）

| 步 | 做什么 |
|---|---|
| 1 | 每一小节都按 **讲解 → 示例 → ✍️ 动手区 → ✅ 自检** 的顺序走 |
| 2 | 示例单元直接 `Shift+Enter` 跑，先看输出；看不懂再回头翻书对应页 |
| 3 | 动手区**关掉答案自己敲**，写完跑自检，看到 `✅` 才算过 |
| 4 | 一关超过 30 分钟先跳过，明天再来。卡太久通常是前置概念没吃透 |

**三条规矩**

1. 「第 0 节 → 自检工具箱」必须先运行，否则 `check()` / `progress()` 没定义。
2. 报错别慌：从**最后一行**往上读，最后一行是错误类型，上面几行是位置。
3. 只跑一部分单元会导致后面 `NameError`——乱了就 `Restart` 内核，从上往下重跑。
4. 练完想从空白重来一遍：用文末「练完一遍，怎么重来」里的重置任务，一键还原。

## 全书地图（10 关 / 27 小节 / 103 个检查点）

| 关 | 小节 | 书的位置 | 周次 |
|---|---|---|---|
| 1 | 变量、字符串、数字（3 节） | 第 2 章 p.60–84 | 第 1 周 |
| 2 | 列表、切片、元组（3 节） | 第 3 章 p.85–105、4.4–4.5 节 | 第 1 周 |
| 3 | if 条件（3 节） | 第 5 章 p.136–161 | 第 1 周 |
| 4 | 字典（3 节） | 第 6 章 p.162–189 | 第 1 周 |
| 5 | 循环与推导式（3 节） | 第 4 章 p.106–135 | 第 2 周 |
| 6 | while 与 input（3 节） | 第 7 章 p.190–209 | 第 2 周 |
| 7 | 函数（3 节） | 第 8 章 p.210–247 | 第 2 周 |
| 8 | 文件与异常（3 节） | 第 10 章 p.282–314 | 第 2 周 |
| 9 | 报错怎么读（1 节） | 贯穿全书 | 第 2 周 |
| 10 | 五个过关练习（2 节） | 手册原文 | 第 2 周 |

> **这两周不学**：类的继承、装饰器、生成器、正则、多线程、Web 框架、算法竞赛题。看到「二叉树」「动态规划」直接跳过——你的目标是处理表格数据，不是通过面试。

In [ ]:
# 第 0 节 · 环境自检：先跑这一格
import sys
import platform

print("Python", sys.version.split()[0], "|", platform.system())
print("解释器 :", sys.executable)

try:
    import numpy

    print("numpy  :", numpy.__version__)
except ImportError:
    print("numpy  : 未安装 → 终端运行 uv add numpy")

In [ ]:
# 第 0 节 · 自检工具箱（不用改，直接运行）
# 用法： check(lambda: 表达式, "这句是提示")
_OK, _BAD = 0, 0


def check(expr, hint=""):
    """expr 传 lambda；通过打 ✅，没写打 ⬜，写错打 ❌。返回 True/False。"""
    global _OK, _BAD
    try:
        passed = bool(expr())
    except NameError as e:
        _BAD += 1
        print(f"⬜ 还没写：{e}　→ 先去上面的动手区写代码")
        return False
    except Exception as e:
        _BAD += 1
        print(f"❌ 运行出错：{type(e).__name__}: {e}")
        return False
    if passed:
        _OK += 1
        print("✅ 通过" + (f"：{hint}" if hint else ""))
    else:
        _BAD += 1
        print("❌ 未通过" + (f"：{hint}" if hint else ""))
    return passed


def progress():
    total = _OK + _BAD
    star = "🎉 全对，可以进下一节" if total and _BAD == 0 else ""
    print(f"—— 累计：✅ {_OK} / {total}　{star}")


print("工具箱就绪：check() / progress() 可用")

In [ ]:
# 第 0 节 · 数据兜底（把笔记发给别人、或在浏览器里跑的时候用）
# 本机有 data/ 就什么也不做；没有就从 GitHub 下载练习数据。
import urllib.request
from pathlib import Path

RAW = "https://raw.githubusercontent.com/crane-DZ/python_learningApp/main/"
FILES = [
    "data/experiments_raw.csv",
    "data/hello.txt",
    "data/learning_python.txt",
    "data/runs/run01.txt",
    "data/runs/run02.txt",
    "data/runs/run03.txt",
]

if Path("data/experiments_raw.csv").exists():
    print("data/ 已存在，跳过下载")
else:
    print("没有 data/，尝试从 GitHub 下载……")
    for rel in FILES:
        try:
            Path(rel).parent.mkdir(parents=True, exist_ok=True)
            with urllib.request.urlopen(RAW + rel, timeout=20) as r:
                Path(rel).write_bytes(r.read())
            print("  ok", rel)
        except Exception as e:
            print("  失败", rel, "→", type(e).__name__, e)
    print("如果全部失败：确认仓库已公开，或把 data/ 文件夹一起复制过来。")

# 第 1 关 · 变量与字符串

📖 **书 p.60–84（第 2 章）**　🎯 第 1 周（第 1–3 天）

过关标准：能建变量、能拼 f-string、会用 `strip()` 去空白、知道 `7/2` 和 `7//2` 的区别。

## 1.1 变量、print 与 f-string

📖 **书 2.1–2.3 节**（p.61–73）　✍️ 练习 2.1–2.3

变量就是给一个值起个标签。名字要能说出含义（`temp_c` 而不是 `t`），单词之间用下划线，不能以数字开头，区分大小写。

| 写法 | 结果 |
|---|---|
| `print("你好")` | 原样输出 |
| `x = 5` | 建变量，不输出任何东西 |
| `f"收率 {y:.1f}%"` | 把变量塞进字符串，`:.1f` 表示保留一位小数 |

**最常见的一个坑**：`print("a" + 5)` 会报 `TypeError`——字符串不能直接和数字相加，用 f-string。

In [ ]:
message = "跟着书学 Python"
print(message)

message = "改一条消息再打印"
print(message)

my_name = "小墨"
intro = f"我是{my_name}，正在学 Python"
print(intro)
print("这句话有", len(intro), "个字符")

rate = 62.36
print(f"收率 {rate:.1f}%")

In [ ]:
# ✍️ 动手区 1.1（自己敲，别复制；写完跑下面的自检）
# 1. 建变量 greeting，值是 你好，Python，然后 print 它
# 2. 建变量 day，先赋值 第 1 天，再改成 第 2 天，最后 print（自检只看最后的值）
# 3. 用 f-string 拼一句话存到 intro_msg，形如：我是小墨，今年开始学 Python（名字用变量 name = "小墨"）
# 在这里写：

In [ ]:
# ✅ 自检 1.1
check(lambda: greeting == "你好，Python", "greeting")
check(lambda: day == "第 2 天", "day 是最后赋的值")
check(lambda: isinstance(intro_msg, str) and "小墨" in intro_msg and "Python" in intro_msg, "intro_msg 里要有名字和 Python")
progress()

## 1.2 字符串方法：大小写与空白

📖 **书 2.3.1–2.3.6 节**（p.67–75）　✍️ 练习 2.4–2.8

| 方法 | 作用 | 例子 |
|---|---|---|
| `.title()` | 每个单词首字母大写 | `"eric matthes"` → `Eric Matthes` |
| `.upper()` / `.lower()` | 全大写 / 全小写 | 比大小写不敏感时先 `lower()` |
| `.strip()` | 去掉首尾空白（含空格、`\t`、`\n`） | 仪器导出的编号常带空白 |
| `.lstrip()` / `.rstrip()` | 只去左边 / 只去右边 | |
| `.removeprefix()` / `.removesuffix()` | 去前缀 / 去后缀 | 拆文件名最常用 |

**注意**：这些方法都**返回新字符串**，不会改原变量。`name.upper()` 单独写一行等于什么也没干。

In [ ]:
name = "eric matthes"
print(name.title(), "|", name.upper(), "|", name.lower())

raw = "  \t张三 \n"
print("原始:", repr(raw))
print("strip 后:", repr(raw.strip()))

filename = "python_notes.txt"
print(filename.removesuffix(".txt"), "|", filename.removeprefix("python_"))

print(name, "← 原变量没被 upper 改掉")

In [ ]:
# ✍️ 动手区 1.2（自己敲，别复制；写完跑下面的自检）
# 1. name2 = "ada lovelace"，用 title() 存到 name_title，用 upper() 存到 name_upper
# 2. raw_name = "\t张三\n"，用 strip() 存到 clean_name
# 3. filename = "python_notes.txt"，用 removesuffix() 去掉 .txt 存到 stem
# 在这里写：

In [ ]:
# ✅ 自检 1.2
check(lambda: name_title == "Ada Lovelace", "name_title")
check(lambda: name_upper == "ADA LOVELACE", "name_upper")
check(lambda: clean_name == "张三", "clean_name 没有空白")
check(lambda: stem == "python_notes", "stem")
progress()

## 1.3 数字、运算与注释

📖 **书 2.4–2.5 节**（p.76–81）　✍️ 练习 2.9–2.11

| 运算 | 结果 | 说明 |
|---|---|---|
| `7 / 2` | `3.5` | 真除法，结果一定是浮点 |
| `7 // 2` | `3` | 整除，只要整数部分 |
| `7 % 2` | `1` | 取余，判奇偶、判倍数用它 |
| `2 ** 10` | `1024` | 幂 |
| `1_000_000` | `1000000` | 下划线只是给人看的 |

**浮点数不能用 `==` 判等**：`0.1 + 0.2` 结果是 `0.30000000000000004`。判等写 `abs(a - b) < 1e-9`。注释用 `#`，写「为什么」而不是「是什么」。

In [ ]:
print("7 / 2 =", 7 / 2, "| 7 // 2 =", 7 // 2, "| 7 % 2 =", 7 % 2)
print("2 ** 10 =", 2 ** 10, "| 1_000_000 =", 1_000_000)
print("3 * 0.1 =", 3 * 0.1)
print("0.1 + 0.2 == 0.3 →", 0.1 + 0.2 == 0.3)
print("容差判等 →", abs(0.1 + 0.2 - 0.3) < 1e-9)
print(round(3.14159, 2), "| 整数除法取整:", 17 // 5, "| 余数:", 17 % 5)

In [ ]:
# ✍️ 动手区 1.3（自己敲，别复制；写完跑下面的自检）
# 1. 书里练习 2.9：用加法、减法、乘法、除法各写一个结果是 8 的表达式，放进列表 eight（4 个元素）
# 2. a = 17，求它除以 5 的商存到 q、余数存到 r
# 3. fav = 7，用 f-string 写一句含数字 7 的话存到 fav_msg
# 4. 浮点陷阱：trap = (0.1 + 0.2 == 0.3)，close = abs(0.1 + 0.2 - 0.3) < 1e-9
# 在这里写：

In [ ]:
# ✅ 自检 1.3
check(lambda: len(eight) == 4 and all(v == 8 for v in eight), "eight 里 4 个表达式都等于 8")
check(lambda: q == 3 and r == 2, "17 // 5 = 3，17 % 5 = 2")
check(lambda: "7" in fav_msg, "fav_msg 里要有数字 7")
check(lambda: trap is False and close is True, "浮点误差：直接判等是 False，容差判等是 True")
progress()

# 第 2 关 · 列表

📖 **书 p.85–105（第 3 章）、p.123–131（4.4–4.5 节）**　🎯 第 1 周（第 4–6 天）

过关标准：能用索引和切片取元素、能增删改、知道 `sorted()` 和 `.sort()` 的区别、知道 `b = a` 不是复制。

## 2.1 索引与增删改

📖 **书 3.1–3.2 节**（p.86–98）　✍️ 练习 3.1–3.7

| 操作 | 写法 | 注意 |
|---|---|---|
| 取值 | `guests[0]`、`guests[-1]` | 从 0 开始，`-1` 是最后一个 |
| 改 | `guests[1] = "李四四"` | 索引存在才能改 |
| 加 | `append(x)` 加到末尾、`insert(0, x)` 插到指定位置 | |
| 删 | `del guests[0]`、`pop()` 弹出末尾、`remove(值)` 按值删 | `pop` 有返回值 |
| 数数 | `len(guests)` | |

**最容易懵的一段**：`remove()` 只删第一个匹配值；`pop()` 不写索引就删最后一个。

In [ ]:
guests_demo = ["张三", "李四", "王五"]
print("初始:", guests_demo)

guests_demo[1] = "李四四"
guests_demo.append("钱七")
guests_demo.insert(0, "赵六")
print("改完:", guests_demo)
print("第一个:", guests_demo[0], "| 最后一个:", guests_demo[-1])

popped_demo = guests_demo.pop()
print("pop 出来的是:", popped_demo)
guests_demo.remove("王五")
print("remove 之后:", guests_demo)

In [ ]:
# ✍️ 动手区 2.1（自己敲，别复制；写完跑下面的自检）
# 1. 建 guests = ["张三", "李四", "王五"]；把 李四 改成 李四四；append("钱七")；insert(0, "赵六")
# 2. 用 pop() 弹出最后一个，返回值存到 popped
# 3. 用 len() 数一下现在的 guests，存到 n_guests
# 在这里写：

In [ ]:
# ✅ 自检 2.1
check(lambda: guests == ["赵六", "张三", "李四四", "王五"], "guests 的最终内容")
check(lambda: popped == "钱七", "popped 是 pop 的返回值")
check(lambda: n_guests == 4, "n_guests")
progress()

## 2.2 排序与长度

📖 **书 3.3 节**（p.99–104）　✍️ 练习 3.8–3.10

| 写法 | 结果 | 原列表 |
|---|---|---|
| `sorted(x)` | 返回排好序的**新**列表 | 不变 |
| `x.sort()` | 原地排序 | 变 |
| `sorted(x, reverse=True)` | 倒序 | 不变 |
| `x.reverse()` | 原地反转 | 变 |

**一个硬规矩**：只想看看排序结果、后面还要用原顺序，就用 `sorted()`。用 `x.sort()` 把原顺序弄丢，是新手最常见的「莫名出错」来源。

In [ ]:
places_demo = ["tokyo", "beijing", "paris"]
print("sorted():", sorted(places_demo))
print("原列表 :", places_demo)

places_demo.sort()
print("sort() 后:", places_demo)
print("倒序:", sorted(places_demo, reverse=True), "| 长度:", len(places_demo))

In [ ]:
# ✍️ 动手区 2.2（自己敲，别复制；写完跑下面的自检）
# 1. 建 places = ["tokyo", "beijing", "paris", "london"]；用 sorted() 排好序存到 sorted_places
# 2. 用 sorted(..., reverse=True) 存到 desc_places
# 3. 用 len() 存到 n_places，并确认 places 本身没被改动
# 在这里写：

In [ ]:
# ✅ 自检 2.2
check(lambda: sorted_places == ["beijing", "london", "paris", "tokyo"], "sorted_places")
check(lambda: desc_places == ["tokyo", "paris", "london", "beijing"], "desc_places")
check(lambda: places == ["tokyo", "beijing", "paris", "london"], "原 places 没被改动")
check(lambda: n_places == 4, "n_places")
progress()

## 2.3 切片、复制与元组

📖 **书 4.4–4.5 节**（p.123–131）　✍️ 练习 4.10–4.13

| 写法 | 意思 |
|---|---|
| `x[:3]` | 前三个（索引 0、1、2） |
| `x[-2:]` | 后两个 |
| `x[1:3]` | 索引 1、2（**左闭右开**） |
| `y = x[:]` | 复制一份，改 y 不影响 x |
| `t = (1, 2)` | 元组，**不能改**，用来表示「不该变的东西」 |

**`y = x` 不是复制**，两个名字指向同一份数据。改 y 等于改 x——书 4.4.3 节专门警告过。

In [ ]:
foods_demo = ["pizza", "falafel", "carrot cake", "cannoli"]
print("前三个:", foods_demo[:3])
print("后两个:", foods_demo[-2:])

friend_demo = foods_demo[:]
friend_demo.append("ice cream")
print("我的:", foods_demo)
print("朋友的:", friend_demo)

point = (3, 5)
print("元组:", point, "| 第一个:", point[0])
try:
    point[0] = 9
except TypeError as e:
    print("元组不能改 →", type(e).__name__)

In [ ]:
# ✍️ 动手区 2.3（自己敲，别复制；写完跑下面的自检）
# 1. 建 my_foods = ["pizza", "falafel", "carrot cake", "cannoli"]；切片取前三个存 first3、后两个存 last2
# 2. 用 [:] 复制到 friend_foods 并 append("ice cream")；用 len() 存 my_len，确认我这份没变
# 3. 元组 menu = ("红烧肉", "清蒸鱼", "炒青菜")，试着改一个元素并捕获异常，把异常类型名存到 tpl_err
# 在这里写：

In [ ]:
# ✅ 自检 2.3
check(lambda: first3 == ["pizza", "falafel", "carrot cake"], "first3")
check(lambda: last2 == ["carrot cake", "cannoli"], "last2")
check(lambda: friend_foods[-1] == "ice cream" and my_len == 4 and "ice cream" not in my_foods, "复制之后改朋友的，我这份不受影响")
check(lambda: tpl_err == "TypeError", "元组改不了，异常类型是 TypeError")
progress()

# 第 3 关 · if 条件判断

📖 **书 p.136–161（第 5 章）**　🎯 第 1 周（第 6–7 天）

过关标准：能写条件测试、会用 `elif` 且知道顺序为什么重要、会用 `if` 处理空列表。

## 3.1 条件测试与布尔值

📖 **书 5.1–5.2 节**（p.137–145）　✍️ 练习 5.1–5.2

| 写法 | 结果 |
|---|---|
| `5 > 3` | `True` |
| `"a" == "A"` | `False`（区分大小写） |
| `x != y` | 不相等 |
| `a and b` / `a or b` / `not a` | 组合 |
| `x in 列表` / `x not in 列表` | 在不在里面 |
| `if 列表:` | 非空为真，空列表为假 |

**注意**：`=` 是赋值，`==` 是判断。写错成 `if x = 5:` 直接 `SyntaxError`。

In [ ]:
print(5 > 3, 5 < 3, "a" == "A", "a" == "a")
print(10 >= 10, 7 != 7, True and False, True or False)

fruits = ["apple", "banana"]
print("apple in fruits →", "apple" in fruits)
print("x not in 'hello' →", "x" not in "hello")
print("空列表为假 →", bool([]), "| 非空为真 →", bool(fruits))

In [ ]:
# ✍️ 动手区 3.1（自己敲，别复制；写完跑下面的自检）
# 1. 按顺序写 5 个条件测试，结果放进列表 tests：5>3、5<3、"a"=="a"、"a"!="b"、10>=10
# 2. 判断 5 在不在 [1, 2, 3] 里，结果存到 has_five
# 3. 判断 "x" 不在 "hello" 里，结果存到 no_x
# 在这里写：

In [ ]:
# ✅ 自检 3.1
check(lambda: tests == [True, False, True, True, True], "tests 五个结果")
check(lambda: has_five is False, "has_five")
check(lambda: no_x is True, "no_x")
progress()

## 3.2 if-elif-else 与多分支

📖 **书 5.3 节**（p.146–154）　✍️ 练习 5.3–5.7

`if / elif / else` **只走第一个成立的分支**，后面的条件根本不会判断。所以顺序决定结果：

```python
if y >= 60: return "合格"
elif y >= 90: return "优"      # 永远到不了：第一条已经把它接住了
```

**规则**：条件宽的往后写，窄的往前写。写完拿边界值试一下（89.9、90、59.9）。

In [ ]:
def score_demo(alien_color):
    if alien_color == "green":
        return 5
    elif alien_color == "yellow":
        return 10
    elif alien_color == "red":
        return 15
    return 0


for c in ["green", "yellow", "red", "blue"]:
    print(c, "→", score_demo(c))

In [ ]:
# ✍️ 动手区 3.2（自己敲，别复制；写完跑下面的自检）
# 1. 写 score(color)：green 得 5 分、yellow 得 10 分、red 得 15 分、其他得 0 分（照上面示例改）
# 2. 写 stage(age)：<2 婴儿、<4 幼儿、<13 儿童、<20 青少年、<65 成年人、其余 老年人
# 3. fruits = ["apple", "banana"]，用 in 判断 apple 在不在，结果存到 has_apple
# 在这里写：

In [ ]:
# ✅ 自检 3.2
check(lambda: score("green") == 5 and score("yellow") == 10 and score("red") == 15, "三种颜色的分数")
check(lambda: score("blue") == 0, "其他颜色得 0 分")
check(lambda: stage(1) == "婴儿" and stage(3) == "幼儿" and stage(12) == "儿童", "年龄小的三段")
check(lambda: stage(18) == "青少年" and stage(30) == "成年人" and stage(70) == "老年人", "年龄大的三段")
check(lambda: has_apple is True, "has_apple")
progress()

## 3.3 用 if 处理列表

📖 **书 5.4 节**（p.155–159）　✍️ 练习 5.8–5.11

三件事在真实代码里天天出现：

1. **特殊元素特殊对待**：`if user == "admin":` 单独给管理员一条消息。
2. **先判断列表非空**：`if users:` 否则 `users[0]` 直接 `IndexError`。
3. **一个列表、多条独立 `if`**：每件元素都要检查，就不要用 `elif`。

`ordinal`（第几）这种题最能练 `elif` 顺序：1st / 2nd / 3rd 要排在 `th` 前面。

In [ ]:
users = ["admin", "tom", "jerry"]
greetings_demo = []
for user in users:
    if user == "admin":
        greetings_demo.append("Hello admin, 要看状态报告吗？")
    else:
        greetings_demo.append(f"Hello {user}")

print(greetings_demo)

empty_demo = []
if empty_demo:
    print("有用户")
else:
    print("We need to find some users!")

In [ ]:
# ✍️ 动手区 3.3（自己敲，别复制；写完跑下面的自检）
# 1. 建 users = ["admin", "tom", "jerry"]，遍历生成 greetings 列表：admin 那条用 "Hello admin, 要看状态报告吗？"，其余 "Hello <名字>"
# 2. 建 empty_users = []，用 if 判断，把 "We need to find some users!" 存到 msg_empty
# 3. 序数：1 到 9，生成 ordinals 列表（"1st" "2nd" "3rd" "4th" … "9th"）
# 在这里写：

In [ ]:
# ✅ 自检 3.3
check(lambda: len(greetings) == 3 and greetings[0].startswith("Hello admin"), "greetings 第一条是管理员消息")
check(lambda: "tom" in greetings[1], "第二条是给 tom 的")
check(lambda: msg_empty == "We need to find some users!", "msg_empty")
check(lambda: ordinals == ["1st", "2nd", "3rd", "4th", "5th", "6th", "7th", "8th", "9th"], "ordinals 九个都对")
progress()

# 第 4 关 · 字典

📖 **书 p.162–189（第 6 章）**　🎯 第 1 周（第 7–8 天）

过关标准：会增删改查、知道 `get()` 和 `[]` 的区别、能遍历键值、能写嵌套结构。

## 4.1 字典的基本操作

📖 **书 6.1–6.2 节**（p.163–172）　✍️ 练习 6.1–6.3

字典是「一个样本的全部字段」，也是 pandas 一行的雏形。

| 操作 | 写法 | 坑 |
|---|---|---|
| 建 | `{"id": "S-03", "y": 62.4}` | 键要加引号 |
| 取值 | `d["y"]` | 键不存在 → `KeyError` |
| 安全取 | `d.get("y", 默认值)` | 读仪器字段一律用 `get` |
| 加/改 | `d["note"] = "含杂质"` | 键在就改，不在就加 |
| 删 | `del d["note"]`、`d.pop("note")` | `pop` 有返回值 |
| 键数 | `len(d)` | 不是元素个数 |

In [ ]:
person_demo = {"first_name": "San", "last_name": "Zhang", "age": 20, "city": "Beijing"}
print(person_demo, "| 年龄:", person_demo["age"])

person_demo["job"] = "student"
person_demo["age"] = 21
print("加/改后:", person_demo)

print("有 email 吗:", person_demo.get("email"))
print("带默认值:", person_demo.get("email", "未填"))

removed_demo = person_demo.pop("job")
print("pop 出来:", removed_demo, "| 现在键数:", len(person_demo))

In [ ]:
# ✍️ 动手区 4.1（自己敲，别复制；写完跑下面的自检）
# 1. 建 person = {"first_name": "San", "last_name": "Zhang", "age": 20, "city": "Beijing"}；加 job = "student"；把 age 改成 21
# 2. 用 get 取 email（默认 "未填"）存到 email
# 3. 用 pop 删掉 job，返回值存到 removed
# 4. 用 len 存到 n_keys
# 在这里写：

In [ ]:
# ✅ 自检 4.1
check(lambda: person["age"] == 21 and "job" not in person, "age 改成 21，job 已删")
check(lambda: email == "未填", "email 用了默认值")
check(lambda: removed == "student", "removed 是 pop 的返回值")
check(lambda: n_keys == 4, "n_keys（first_name、last_name、age、city）")
progress()

## 4.2 遍历字典

📖 **书 6.3 节**（p.173–180）　✍️ 练习 6.4–6.6

| 写法 | 拿到什么 |
|---|---|
| `for k, v in d.items():` | 键和值（最常用） |
| `for k in d.keys():` | 只要键 |
| `for k in sorted(d.keys()):` | 按字母顺序的键 |
| `for v in d.values():` | 只要值（可能有重复） |

**记住**：默认遍历 `for k in d:` 就是遍历键。要按顺序输出，键先 `sorted()` 一下。

In [ ]:
person_demo = {"first_name": "San", "last_name": "Zhang", "age": 21, "city": "Beijing"}

for k, v in person_demo.items():
    print(k, "=", v)

print("排序后的键:", sorted(person_demo.keys()))
print("所有值:", list(person_demo.values()))

In [ ]:
# ✍️ 动手区 4.2（自己敲，别复制；写完跑下面的自检）
# 1. 用 items() 遍历 person，把每一项拼成 "键=值" 放进列表 lines
# 2. 用 sorted() 排好键，存到 sorted_keys
# 3. 练习 6.6：favorite_languages = {"张三": "python", "李四": "c"}，名单 names = ["张三", "王五"]；已参加的存 answered、没参加的存 not_answered
# 在这里写：

In [ ]:
# ✅ 自检 4.2
check(lambda: len(lines) == 4 and lines[0].startswith("first_name="), "lines 四条，形如 键=值")
check(lambda: sorted_keys == sorted(person.keys()), "sorted_keys")
check(lambda: answered == ["张三"] and not_answered == ["王五"], "answered / not_answered")
progress()

## 4.3 嵌套：字典列表与字典里的字典

📖 **书 6.4 节**（p.181–188）　✍️ 练习 6.7–6.11

两种嵌套形态，读法完全不同：

```python
people = [{"name": "张三", "age": 20}, {"name": "李四", "age": 25}]   # 列表里放字典：一行一个样本
cities = {"beijing": {"population": 2100, "country": "中国"}}          # 字典里放字典：一个键一套属性
```

取法：`people[0]["age"]`、`cities["beijing"]["population"]`。数不清有几层时，先 `print` 出来看一眼。

In [ ]:
people_demo = [
    {"name": "张三", "age": 20},
    {"name": "李四", "age": 25},
]
print(people_demo[0]["name"], people_demo[1]["age"])

ages = [p["age"] for p in people_demo]
print("年龄:", ages, "| 平均:", sum(ages) / len(ages))

cities_demo = {
    "beijing": {"country": "中国", "population": 2100},
    "tokyo": {"country": "日本", "population": 1400},
}
print(cities_demo["beijing"]["population"])

In [ ]:
# ✍️ 动手区 4.3（自己敲，别复制；写完跑下面的自检）
# 1. 建 people 列表，三个字典：张三 20 岁、李四 25 岁、王五 30 岁；算平均年龄存到 avg_age
# 2. 建 cities 字典，两个城市各带 country/population；取 beijing 的 population 存到 bj_pop
# 在这里写：

In [ ]:
# ✅ 自检 4.3
check(lambda: len(people) == 3 and abs(avg_age - 25.0) < 1e-9, "avg_age = (20+25+30)/3")
check(lambda: bj_pop == 2100, "bj_pop")
progress()

# 第 5 关 · 循环与推导式

📖 **书 p.106–135（第 4 章）**　🎯 第 2 周（第 1–3 天）

过关标准：能写 `for` 遍历、知道缩进就是语法、能用列表推导式一行顶三行。

## 5.1 for 循环与 range()

📖 **书 4.1–4.3 节**（p.107–122）　✍️ 练习 4.1–4.7

| 写法 | 拿到什么 |
|---|---|
| `for x in 列表:` | 逐个元素 |
| `for i in range(5):` | 0,1,2,3,4（不含 5） |
| `for i in range(1, 6):` | 1..5 |
| `for i in range(1, 21, 2):` | 1,3,5,…,19（第三步长） |
| `for i in range(20, 0, -1):` | 倒着数 |

`range()` **左闭右开**：`range(1, 21)` 才包含 20。这是这两周最常见的差一错误。

In [ ]:
for fruit in ["apple", "banana"]:
    print("我喜欢", fruit)
print("循环结束后，fruit 变量还留着最后一个值:", fruit)

for i in range(3):
    print("i =", i)

print("1..20:", list(range(1, 21)))
print("奇数:", list(range(1, 21, 2)))

In [ ]:
# ✍️ 动手区 5.1（自己敲，别复制；写完跑下面的自检）
# 1. 用 for + range 生成列表 nums_20，内容是 1 到 20
# 2. 对 nums_20 求和存 total_sum，求最小存 min_n、最大存 max_n
# 3. 用 range 第三步长生成 1 到 20 的奇数列表 odds
# 4. 生成 3 到 30 之间能被 3 整除的数，存到 threes
# 在这里写：

In [ ]:
# ✅ 自检 5.1
check(lambda: nums_20 == list(range(1, 21)), "nums_20 是 1..20")
check(lambda: total_sum == 210 and min_n == 1 and max_n == 20, "和 210、最小 1、最大 20")
check(lambda: odds == list(range(1, 21, 2)), "odds 是 1..20 的奇数")
check(lambda: threes == list(range(3, 31, 3)), "threes 是 3..30 的 3 的倍数")
progress()

## 5.2 缩进就是语法（以及怎么读缩进错误）

📖 **书 4.2、4.6 节**（p.112–115、132–134）　✍️ 练习 4.14

Python 用**缩进**表示从属关系，不用大括号。所以缩进不是「排版好看」，是语法的一部分。

| 错误 | 意思 |
|---|---|
| `IndentationError: expected an indented block` | 该缩进的地方没缩进 |
| `IndentationError: unexpected indent` | 不该缩进的地方多缩了 |
| `TabError` | 空格和 Tab 混用了 |

规矩：**只用 4 个空格**，别按 Tab。VS Code 里按 Tab 会自动转成空格，但复制来的代码可能带 Tab。

In [ ]:
good = "for i in range(3):\n    print(i)\n"
exec(compile(good, "<good>", "exec"))

bad = "for i in range(3):\nprint(i)\n"
try:
    compile(bad, "<bad>", "exec")
except IndentationError as e:
    print("捕获到:", type(e).__name__, "→", e.msg)

mixed = "for i in range(2):\n    print(i)\n\tprint('tab')\n"
try:
    compile(mixed, "<mixed>", "exec")
except Exception as e:
    print("混用 Tab:", type(e).__name__)

In [ ]:
# ✍️ 动手区 5.2（自己敲，别复制；写完跑下面的自检）
# 1. 下面是缩进写错的一段代码：bad_indent = "for i in range(3):\nprint(i)\n"。用 try/except 编译它，把异常类型名存到 err_indent
# 2. 把缩进修好的版本写成字符串 fixed_code，用 compile() 验证能通过，然后把 fixed_ok 设成 True
# 在这里写：

In [ ]:
# ✅ 自检 5.2
check(lambda: err_indent == "IndentationError", "err_indent")
check(lambda: fixed_ok is True, "fixed_ok")
progress()

## 5.3 数值列表与列表推导式

📖 **书 4.3.3–4.3.4 节**（p.120–122）　✍️ 练习 4.6–4.9

推导式是「用一行写一个循环 + 一次追加」：

```python
squares = [i ** 2 for i in range(1, 11)]            # 前 10 个平方
evens   = [i for i in range(1, 21) if i % 2 == 0]    # 加筛选条件
```

读法：从左往右，「把 `i ** 2` 放进列表，`i` 来自 `range(1, 11)`」。
**别一开始就写三四层的推导式**——可读性比短更重要，嵌套两层以上就用普通 for。

In [ ]:
squares_demo = [i ** 2 for i in range(1, 6)]
print("平方:", squares_demo)

cubes_demo = [i ** 3 for i in range(1, 6)]
print("立方:", cubes_demo)

evens_demo = [i ** 2 for i in range(1, 11) if i % 2 == 0]
print("偶数平方:", evens_demo)

print("等价写法:", [i * i for i in range(1, 6)])

In [ ]:
# ✍️ 动手区 5.3（自己敲，别复制；写完跑下面的自检）
# 1. 用推导式生成 1 到 10 的平方，存到 squares
# 2. 用推导式生成 1 到 10 的立方，存到 cubes
# 3. 用带 if 的推导式生成 1 到 20 中偶数的平方，存到 evens_sq
# 在这里写：

In [ ]:
# ✅ 自检 5.3
check(lambda: squares == [i ** 2 for i in range(1, 11)], "squares")
check(lambda: cubes == [i ** 3 for i in range(1, 11)], "cubes")
check(lambda: evens_sq == [i ** 2 for i in range(1, 21) if i % 2 == 0], "evens_sq")
progress()

# 第 6 关 · while 与 input

📖 **书 p.190–209（第 7 章）**　🎯 第 2 周（第 3–5 天）

过关标准：知道 `input()` 拿到的是字符串、会用 `while/break/continue`、能用 while 处理列表。

## 6.1 input() 与类型转换

📖 **书 7.1 节**（p.191–195）　✍️ 练习 7.1–7.3

**`input()` 拿到的永远是字符串**，哪怕用户输入的是 42。要算数必须先 `int()` 或 `float()`。

```python
age = input("你几岁？")      # 用户输入 20 → age 是 "20"
print(age + 1)               # ✗ TypeError，字符串不能加数字
age = int(age)               # ✓ 先转
print(age + 1)               # 21
```

笔记本里 `input()` 会弹框并**阻塞**，`Run All` 会卡住。所以下面用 `fake_input()` 模拟：真实 `.py` 脚本里照书用 `input()`。

In [ ]:
def fake_input(prompt, value):
    """模拟 input()：把用户输入固定成 value，避免笔记本阻塞。"""
    print(prompt, value, "（模拟输入）")
    return value


reply = fake_input("数量：", "42")
print("拿到的是字符串:", repr(reply), "｜转成整数后 +8 =", int(reply) + 8)

people = fake_input("几个人吃饭？", "10")
people = int(people)
if people > 8:
    print("没有空桌")
else:
    print("有空桌")

In [ ]:
# ✍️ 动手区 6.1（自己敲，别复制；写完跑下面的自检）
# 1. 用 fake_input("数量：", "42") 拿输入，转成整数后加 8，存到 result
# 2. 用 fake_input("几个人？", "10") 拿人数，转成整数；大于 8 时 msg = "没有空桌"，否则 "有空桌"
# 3. 用 fake_input("一个数：", "30") 拿数，判断它是不是 10 的倍数，结果存到 is_mult10
# 在这里写：

In [ ]:
# ✅ 自检 6.1
check(lambda: result == 50, "42 + 8 = 50")
check(lambda: msg == "没有空桌", "10 个人 → 没有空桌")
check(lambda: is_mult10 is True, "30 是 10 的倍数")
progress()

## 6.2 while 循环、break 与 continue

📖 **书 7.2 节**（p.196–203）　✍️ 练习 7.4–7.7

`while` 用在「不知道要循环几次」的时候。三件套：

| 关键字 | 作用 |
|---|---|
| `while 条件:` | 条件为真就继续 |
| `break` | 立刻跳出整个循环 |
| `continue` | 跳过本次剩下的代码，回到条件判断 |

**死循环** = 条件永远为真。检查两件事：循环里有没有改那个条件变量？有没有 `break`？笔记本卡住就点单元左边的 ⏹。

In [ ]:
n = 1
total = 0
while n <= 10:
    total += n
    n += 1
print("1..10 求和:", total)

i = 0
while True:
    i += 1
    if i > 3:
        break
    print("i =", i)

j = 0
while j < 6:
    j += 1
    if j % 2 == 0:
        continue
    print("奇数次:", j)

In [ ]:
# ✍️ 动手区 6.2（自己敲，别复制；写完跑下面的自检）
# 1. 用 while 累加 1 到 10，结果存到 total_while
# 2. 用 while + break：从 1 开始逐个累加，和一旦超过 100 就停；把加到第几个数存到 n_break、此时的和存到 sum_break
# 3. 用 while + continue：求 1 到 20 里能被 3 整除的数的和，存到 sum3
# 在这里写：

In [ ]:
# ✅ 自检 6.2
check(lambda: total_while == 55, "1..10 求和 = 55")
check(lambda: n_break == 14 and sum_break == 105, "加到 14 时和首次超过 100（1..14 = 105）")
check(lambda: sum3 == 63, "3+6+9+12+15+18 = 63")
progress()

## 6.3 用 while 处理列表和字典

📖 **书 7.3 节**（p.204–208）　✍️ 练习 7.8–7.10

两个真实套路：

1. **在两个列表之间搬元素**：`while 待处理: 弹出 → 处理后追加到另一个列表`。搬完一个空、一个满。
2. **删除所有等于某个值的元素**：`while 值 in 列表: 列表.remove(值)`。用 `for` 边遍历边删会漏元素。

In [ ]:
orders_demo = ["牛肉", "鸡蛋", "鸡肉"]
finished_demo = []
while orders_demo:
    s = orders_demo.pop(0)
    finished_demo.append(s)
print("做完:", finished_demo, "| 待做:", orders_demo)

orders = ["pastrami", "牛肉", "pastrami", "鸡蛋"]
while "pastrami" in orders:
    orders.remove("pastrami")
print("去掉 pastrami 后:", orders)

responses_demo = {}
pairs = [("张三", "北京"), ("李四", "上海")]
k = 0
while k < len(pairs):
    name, city = pairs[k]
    responses_demo[name] = city
    k += 1
print("调查结果:", responses_demo)

In [ ]:
# ✍️ 动手区 6.3（自己敲，别复制；写完跑下面的自检）
# 1. sandwich_orders = ["A", "B", "C"]；用 while 每次 pop(0) 搬到 finished，直到搬完
# 2. orders2 = ["pastrami", "牛肉", "pastrami", "鸡蛋"]；用 while 删掉所有 pastrami
# 3. 用 while 把 [("张三", "北京"), ("李四", "上海")] 填进字典 responses
# 在这里写：

In [ ]:
# ✅ 自检 6.3
check(lambda: finished == ["A", "B", "C"] and sandwich_orders == [], "搬完：finished 有 3 个，原列表空")
check(lambda: orders2 == ["牛肉", "鸡蛋"], "orders2 里没有 pastrami 了")
check(lambda: len(responses) == 2 and responses["李四"] == "上海", "responses 两个键值对")
progress()

# 第 7 关 · 函数

📖 **书 p.210–247（第 8 章）**　🎯 第 2 周（第 5–7 天）

过关标准：会定义带参数和返回值的函数、知道 `return` 和 `print` 的区别、会把函数放进模块导入。

## 7.1 定义函数、位置实参、默认值

📖 **书 8.1–8.2 节**（p.211–220）　✍️ 练习 8.1–8.5

```python
def make_shirt(size, text="I love Python"):     # text 有默认值
    return f"{size} T 恤，印着：{text}"

make_shirt("大号")                    # 用默认值
make_shirt("中号", text="学 Python")   # 关键字实参，顺序随便
```

**`return` 和 `print` 不是一回事**：`print` 只是显示，`return` 才是把结果交出去。忘了 `return` 的函数返回 `None`，这是新手最隐蔽的 bug。

In [ ]:
def make_shirt_demo(size, text="I love Python"):
    return f"{size} T 恤，印着：{text}"


print(make_shirt_demo("大号"))
print(make_shirt_demo("中号", text="学 Python"))
print(make_shirt_demo(text="学 Python", size="小号"))

def describe_city_demo(city, country="中国"):
    return f"{city} 属于 {country}"


print(describe_city_demo("Beijing"), "|", describe_city_demo("Paris", "法国"))

In [ ]:
# ✍️ 动手区 7.1（自己敲，别复制；写完跑下面的自检）
# 1. 写 display_message()：打印一句本章主题的话；调用它，把返回值存到 r_none
# 2. 写 make_shirt(size, text="I love Python")，返回形如 "大号 T 恤，印着：…" 的字符串
# 3. 写 describe_city(city, country="中国")，返回形如 "Beijing 属于 中国" 的字符串
# 在这里写：

In [ ]:
# ✅ 自检 7.1
check(lambda: r_none is None, "只 print 不 return，返回值是 None")
check(lambda: "大号" in make_shirt("大号") and "I love Python" in make_shirt("大号"), "用默认值做出的 T 恤")
check(lambda: make_shirt("中号", text="学 Python") == "中号 T 恤，印着：学 Python", "关键字实参")
check(lambda: describe_city("Tokyo") == "Tokyo 属于 中国", "默认国家")
check(lambda: describe_city("Paris", "法国") == "Paris 属于 法国", "位置实参")
progress()

## 7.2 返回值：单值、字典、可选参数

📖 **书 8.3 节**（p.222–228）　✍️ 练习 8.6–8.8

返回值可以是任何东西。字段多的时候**返回字典比返回元组清楚**：

```python
def build_person(first, last, **info):
    return {"first": first, "last": last, **info}
```

`**info` 收集所有关键字实参，`*` 开头的是「任意数量」——这两个符号别搞反。

In [ ]:
def city_country_demo(city, country):
    return f"{city}, {country}"


print(city_country_demo("Santiago", "Chile"))


def build_person_demo(first, last, age=None):
    person = {"first": first, "last": last}
    if age is not None:
        person["age"] = age
    return person


print(build_person_demo("San", "Zhang"))
print(build_person_demo("San", "Zhang", age=20))

In [ ]:
# ✍️ 动手区 7.2（自己敲，别复制；写完跑下面的自检）
# 1. 写 city_country(city, country)，返回 "城市, 国家" 格式的字符串
# 2. 写 build_person(first, last, **info)，返回含 first/last 以及 info 里所有键的字典
# 3. 写 make_album(artist, title, songs=None)，返回 {"artist":…, "title":…}，songs 不是 None 时再加这个键
# 在这里写：

In [ ]:
# ✅ 自检 7.2
check(lambda: city_country("Santiago", "Chile") == "Santiago, Chile", "city_country")
check(lambda: build_person("San", "Zhang", age=20)["age"] == 20, "关键字实参进了字典")
check(lambda: set(build_person("San", "Zhang").keys()) == {"first", "last"}, "不传 age 就没有 age 键")
check(lambda: "songs" not in make_album("周杰伦", "范特西"), "不传 songs 就没有这个键")
check(lambda: make_album("周杰伦", "范特西", songs=10)["songs"] == 10, "传了 songs 才有")
progress()

## 7.3 传列表、任意数量的实参、把函数放进模块

📖 **书 8.4–8.6 节**（p.229–245）　✍️ 练习 8.9–8.16

| 写法 | 意思 |
|---|---|
| `f(列表)` | 传进去的是**同一个列表**，函数里改它，外面也变 |
| `f(列表[:])` | 传副本，随便改都不影响外面 |
| `def f(*toppings)` | 收集任意多个位置实参，`toppings` 是元组 |
| `from module import func` | 从模块里导入函数 |

**为什么要拆模块**：函数一多，一个文件几百行就找不着了。拆成 `utils.py` 再 `import`，是第 8 周写 `clean.py` 的前置技能。

In [ ]:
def send_messages_demo(messages):
    sent = []
    while messages:
        sent.append(messages.pop(0))
    return sent


msgs = ["a", "b"]
print("搬走:", send_messages_demo(msgs), "| 原列表:", msgs)


def send_copy_demo(messages):
    sent = []
    for m in messages[:]:
        sent.append(m)
    return sent


msgs2 = ["x", "y"]
print("副本版:", send_copy_demo(msgs2), "| 原列表:", msgs2)


def make_sandwich_demo(*toppings):
    return "三明治里有：" + "、".join(toppings)


print(make_sandwich_demo("生菜", "火腿"))

from math import sqrt

print("sqrt(16) =", sqrt(16))

In [ ]:
# ✍️ 动手区 7.3（自己敲，别复制；写完跑下面的自检）
# 1. 写 send_messages(messages)：用 pop 把消息搬到新列表并返回，原列表会被清空
# 2. 写 send_messages_copy(messages)：用副本遍历，返回新列表但原列表不变
# 3. 写 make_sandwich(*toppings)，返回 "三明治里有：a、b" 这种字符串
# 4. from math import sqrt，算出 sqrt(16) 存到 r_sqrt
# 在这里写：

In [ ]:
# ✅ 自检 7.3
check(lambda: send_messages(["a", "b"]) == ["a", "b"], "返回值是搬走的消息")
check(lambda: send_messages_copy(["x", "y"]) == ["x", "y"], "副本版返回值")
check(lambda: make_sandwich("生菜", "火腿") == "三明治里有：生菜、火腿", "make_sandwich")
check(lambda: r_sqrt == 4.0, "sqrt(16) = 4.0")
progress()

# 第 8 关 · 文件与异常

📖 **书 p.282–314（第 10 章）**　🎯 第 2 周（第 7–9 天）

过关标准：会用 `with open(...)` 读和写、知道要写 `encoding="utf-8"`、会用 `try/except` 兜住缺文件和类型错误。

本关数据都在 `data/` 下：

```
data/
├── experiments_raw.csv   12 行实验记录（批次,温度_C,时间_min,收率_pct）
├── hello.txt             一行打卡
├── learning_python.txt   三行 "In Python you can ..."
└── runs/run01..03.txt    每次运行一个文件，1 行表头 + 3 行数据
```

## 8.1 读文件

📖 **书 10.1 节**（p.283–291）　✍️ 练习 10.1–10.3

| 写法 | 拿到什么 |
|---|---|
| `open(p, encoding="utf-8")` | 文件对象（**必须写 encoding**） |
| `f.read()` | 整个文件一个字符串 |
| `f.read().splitlines()` | 去掉换行的行列表（最常用） |
| `for line in f:` | 逐行，每行带 `\n` |

**Windows 上不写 `encoding="utf-8"` 会按 GBK 解码**，中文直接乱码或 `UnicodeDecodeError`。养成每次 `open` 都带上的习惯。

In [ ]:
from pathlib import Path

print("当前目录:", Path.cwd())
print("data 存在:", Path("data").exists())

with open("data/hello.txt", encoding="utf-8") as f:
    content_demo = f.read()
print("--- hello.txt ---")
print(content_demo.strip())

print("按行拆:", content_demo.splitlines())

with open("data/learning_python.txt", encoding="utf-8") as f:
    for n, line in enumerate(f, 1):
        print(n, line.rstrip())

In [ ]:
# ✍️ 动手区 8.1（自己敲，别复制；写完跑下面的自检）
# 1. 读 data/hello.txt 的全部内容到 content
# 2. 把 content 按行拆开，行列表存到 lines
# 3. 读 data/learning_python.txt，把里面的 Python 全部换成 C，结果存到 swapped（用 replace）
# 在这里写：

In [ ]:
# ✅ 自检 8.1
check(lambda: "打卡" in content, "content 里有打卡那行")
check(lambda: len(lines) == 2, "hello.txt 两行")
check(lambda: "Python" not in swapped and "C" in swapped, "swapped 里的 Python 都换成了 C")
progress()

## 8.2 写文件与追加

📖 **书 10.2 节**（p.292–294）　✍️ 练习 10.4–10.5

| 模式 | 意思 |
|---|---|
| `"w"` | 覆盖写（文件不存在就新建） |
| `"a"` | 追加写 |
| `"r"` | 只读（默认） |

写文件也一律 `encoding="utf-8"`。**`"w"` 会清空原内容**，想保留就用 `"a"`——这两个搞反会丢数据。

In [ ]:
with open("data/demo_write.txt", "w", encoding="utf-8") as f:
    f.write("第一行\n")

with open("data/demo_write.txt", "a", encoding="utf-8") as f:
    f.write("第二行\n")

with open("data/demo_write.txt", encoding="utf-8") as f:
    print("读回来:", f.read().splitlines())

In [ ]:
# ✍️ 动手区 8.2（自己敲，别复制；写完跑下面的自检）
# 1. 把 "张三\n" 用 "w" 模式写进 data/guest.txt，然后读回来存到 guest
# 2. 用 "a" 模式再追加 "李四\n"，然后按行读回来存到 guest_lines
# 在这里写：

In [ ]:
# ✅ 自检 8.2
check(lambda: guest.strip() == "张三", "guest 只有张三")
check(lambda: guest_lines == ["张三", "李四"], "追加之后两行")
progress()

## 8.3 try/except 与 json

📖 **书 10.3–10.4 节**（p.295–313）　✍️ 练习 10.6–10.14

| 异常 | 什么时候出现 |
|---|---|
| `ValueError` | `int("abc")`——值不对 |
| `FileNotFoundError` | 文件不存在 |
| `ZeroDivisionError` | 除以 0 |
| `KeyError` / `IndexError` | 键 / 下标不存在 |

```python
try:
    可能出错的代码
except ValueError:
    出错了怎么办
else:
    没出错才执行的
```

`json.dumps()` 把 Python 对象变成字符串，`json.loads()` 反过来；存文件用 `json.dump(obj, f)` / `json.load(f)`。

In [ ]:
try:
    int("abc")
except ValueError as e:
    print("捕获:", type(e).__name__)

try:
    with open("data/nope.csv", encoding="utf-8") as f:
        f.read()
except FileNotFoundError as e:
    print("捕获:", type(e).__name__)

import json

s = json.dumps({"name": "张三", "fav": 7}, ensure_ascii=False)
print("dumps:", s)
print("loads:", json.loads(s)["fav"])

with open("data/demo.json", "w", encoding="utf-8") as f:
    json.dump({"fav": 7}, f)

with open("data/demo.json", encoding="utf-8") as f:
    print("从文件读回:", json.load(f))

In [ ]:
# ✍️ 动手区 8.3（自己敲，别复制；写完跑下面的自检）
# 1. 用 try/except 处理 int("abc")，把异常类型名存到 err_value
# 2. 写加法程序：用 fake_input 拿 "abc"，转整数时用 try/except 兜住，出错时把 msg_add 设成 "请输入数字"
# 3. 把 7 用 json.dumps 存到 json_str，再用 json.loads 读回存到 fav_num
# 4. 打开 data/nope.csv，用 try/except 捕获，把异常类型名存到 err_file
# 在这里写：

In [ ]:
# ✅ 自检 8.3
check(lambda: err_value == "ValueError", "err_value")
check(lambda: msg_add == "请输入数字", "msg_add")
check(lambda: fav_num == 7, "json 往返之后还是 7")
check(lambda: err_file == "FileNotFoundError", "err_file")
progress()

# 第 9 关 · 报错怎么读

🎯 第 2 周

这一关没有新语法，只有一项技能：**看见红字不慌**。

报错从下往上读：

```
Traceback (most recent call last):
  File "demo.py", line 3, in <module>
    print(1 / 0)
ZeroDivisionError: division by zero
      ↑ 错误类型              ↑ 错误说明
```

最后一行是**错误类型 + 说明**，倒数第二行是**出错的位置和代码**。先读最后一行，再往上找自己写的那行。

## 9.1 五种高频报错，逐个认一遍

📖 **书 贯穿全书**（p.63、p.103、p.304 等）　✍️ —

| 报错 | 典型原因 | 怎么修 |
|---|---|---|
| `NameError` | 变量名打错 / 没定义 | 检查拼写、检查大小写 |
| `IndexError` | 下标超出长度 | `len()` 看一眼，记住从 0 开始 |
| `KeyError` | 字典没这个键 | 改用 `.get(k, 默认值)` |
| `ValueError` | 类型对但值不对 | `int("abc")` 这种，先校验再转 |
| `TypeError` | 类型不对 | 字符串和数字相加、参数个数不对 |

下面四段代码都是「故意写错」的，用 `try/except` 跑一遍，把异常类型名记下来。

In [ ]:
cases = {
    "a": "print(undefined_var)",
    "b": "[1, 2, 3][5]",
    "c": "{'a': 1}['b']",
    "d": "int('abc')",
}

for k, src in cases.items():
    try:
        exec(compile(src, "<%s>" % k, "exec"), {})
    except Exception as e:
        print(k, "→", type(e).__name__)

In [ ]:
# ✍️ 动手区 9.1（自己敲，别复制；写完跑下面的自检）
# 1. code_a = "print(undefined_var)"，跑它并捕获异常，类型名存到 err_a
# 2. code_b = "[1, 2, 3][5]"，同上，存到 err_b
# 3. code_c = "{'a': 1}['b']"，同上，存到 err_c
# 4. code_d = "int('abc')"，同上，存到 err_d
# 在这里写：

In [ ]:
# ✅ 自检 9.1
check(lambda: err_a == "NameError", "err_a")
check(lambda: err_b == "IndexError", "err_b")
check(lambda: err_c == "KeyError", "err_c")
check(lambda: err_d == "ValueError", "err_d")
progress()

# 第 10 关 · 五个过关练习

📖 手册《第 1–2 周》「五个练习，做完就算过关」

这五题都跟你后面建模直接相关，不是抽象语法题。前四题用 `data/` 里的数据，第五题用 numpy。

## 10.1 练习 1–3：读 CSV、手工 groupby

📖 **书 手册原文**（p.283 起）　✍️ 练习 1、3

**练习 1**：读一个 CSV，打印前 5 行、行数、每列的数据类型。体会一下 pandas 之前你得多写多少行。

**练习 3**：用字典统计一列里每个取值出现多少次，并按次数排序。这就是 `groupby` 的手工版。

读出来的**全是字符串**——这正是 pandas 存在的理由。

In [ ]:
import csv

with open("data/experiments_raw.csv", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    hdr = next(reader)
    rows_demo = list(reader)

print("列名:", hdr)
print("行数:", len(rows_demo))
print("前 2 行:")
for r in rows_demo[:2]:
    print("  ", r)
print("注意类型:", repr(rows_demo[0][1]), "← 字符串，不是数字 25")

In [ ]:
# ✍️ 动手区 10.1（自己敲，别复制；写完跑下面的自检）
# 1. 练习 1：读 data/experiments_raw.csv，列名存到 header，数据行存到 data_rows；打印前 5 行、行数
# 2. 练习 3：统计「温度_C」每个值出现几次，按次数从多到少排，结果存到 ranking（形如 [("25", 4), …]）
# 在这里写：

In [ ]:
# ✅ 自检 10.1
check(lambda: header == ["批次", "温度_C", "时间_min", "收率_pct"], "header")
check(lambda: len(data_rows) == 12, "data_rows 12 行")
check(lambda: isinstance(data_rows[0][1], str), "读出来是字符串不是数字")
check(lambda: len(ranking) == 4, "4 个不同温度")
check(lambda: ranking[0][1] == 4 and str(ranking[0][0]) == "25", "25 出现 4 次且排第一")
check(lambda: sum(c for _, c in ranking) == 12, "次数合计 12")
progress()

## 10.2 练习 2、4、5：单位换算、批量读文件、numpy

📖 **书 手册原文**（—）　✍️ 练习 2、4、5

**练习 2**：`unit_convert(value, from_unit, to_unit)`，支持 K/℃、kPa/atm、mg/g。做实验的人一定会用到单位换算，换算错误是最常见的低级错误。

**练习 4**：批量读一个文件夹下所有文件，拼成一张表。用到 `glob`、循环、列表追加——真实工作里最常用的模式。

**练习 5**：numpy 矩阵乘法与转置。为第 6 周线性回归 `y = Xβ` 打底。

In [ ]:
from pathlib import Path

files = sorted(Path("data/runs").glob("*.txt"))
print("找到", len(files), "个文件:", [p.name for p in files])

import numpy as np

A = np.array([[1, 2, 3], [4, 5, 6]])
print("A 的形状:", A.shape)
print("A 乘 2:\n", A * 2)
print("A 的转置形状:", A.T.shape)

In [ ]:
# ✍️ 动手区 10.2（自己敲，别复制；写完跑下面的自检）
# 1. 练习 2：写 my_unit_convert(value, from_unit, to_unit)，支持 K↔C、kPa↔atm、g↔mg，未知单位返回 None
# 2. 练习 4：读 data/runs/ 下所有 .txt，每行转成字典 {run, 批次, 温度_C(float), 收率_pct(float)}，全部放进 all_rows
# 3. 练习 5：A = [[1,2,3],[4,5,6]]，B = [[1,0],[0,1],[2,2]]；C = A @ B 存到 C，AT = A.T 存到 AT
# 在这里写：

In [ ]:
# ✅ 自检 10.2
check(lambda: abs(my_unit_convert(300, "K", "C") - 26.85) < 1e-9, "300 K = 26.85 ℃")
check(lambda: abs(my_unit_convert(1, "atm", "kPa") - 101.325) < 1e-9, "1 atm = 101.325 kPa")
check(lambda: my_unit_convert(2, "g", "mg") == 2000, "2 g = 2000 mg")
check(lambda: my_unit_convert(1, "kg", "g") is None, "未知单位返回 None")
check(lambda: len(all_rows) == 9, "三个文件共 9 行")
check(lambda: all_rows[0]["温度_C"] == 25.0 and isinstance(all_rows[0]["温度_C"], float), "温度已转成 float")
check(lambda: all_rows[0]["run"] == "run01.txt", "带文件名")
check(lambda: C.shape == (2, 2) and C[0, 0] == 7 and C[0, 1] == 8, "C 是 2×2，第一行 [7, 8]")
check(lambda: AT.shape == (3, 2), "AT 是 3×2")
progress()

# 通关之后

## 这两周的产出物（手册的验收标准）

- [ ] 一个 `read_csv.py`：读 CSV、打印行数与前 5 行、打印每列类型
- [ ] 一个 `unit_convert()`：K/℃、kPa/atm、mg/g 三组都测过
- [ ] 一个脚本：把 `data/runs/` 里所有文件拼成一张表
- [ ] 上面所有自检全绿

## 常见报错速查

| 报错 | 真实原因 | 怎么办 |
|---|---|---|
| `IndentationError` | 缩进不一致，Tab 和空格混用 | 全删成 4 个空格 |
| `NameError` | 变量没定义，或本单元没运行过 | 从上面的单元依次往下跑 |
| `IndexError` | 索引超出长度 | `len()` 看一眼，记住从 0 开始 |
| `KeyError` | 字典没这个键 | 改用 `.get(k, 默认值)` |
| `TypeError` | 字符串和数字相加、参数个数不对 | 用 f-string 或 `int()` / `str()` 转 |
| `FileNotFoundError` | 路径不对 | `Path.cwd()` 打印当前目录 |
| `UnicodeDecodeError` | 没写 `encoding="utf-8"` | 每个 `open` 都带上 |

## 下一站

| 目标 | 教材位置 | 什么时候学 |
|---|---|---|
| pandas 整理数据（第 3–4 周） | 《Python 数据科学手册》 | **下一步就是它** |
| matplotlib 画图（第 5 周） | 本书第 15–17 章 p.449 起 | 紧接着 |
| 类与继承 | 第 9 章 p.248–281 | 暂缓（手册表 5：可以暂时不会） |
| pytest 测试 | 第 11 章 p.315–335 | 第 8 周写 `clean.py` 前再看 |

**手册原话**：不要在学语法阶段刷算法题——你的目标是处理表格数据，不是通过面试。

## 练完一遍，怎么重来

| 你想保留 | 怎么做 | 结果 |
|---|---|---|
| 只清打印和运行结果 | 工具栏 **Clear All Outputs**，或内核 **Restart** | 代码还在，自检计数归零 |
| 想从空白重做一遍 | `Ctrl+Shift+P` → **Tasks: Run Task** → **🔄 重置练习（先备份）** | 当前答案先备份到 `notebooks/_练习记录/时间戳.ipynb`，整本还原成空白版 |
| 想对照着看 | 上一步做完，打开 `notebooks/_练习记录/` 里最新的那份 | 左边答案、右边空白 |

重置之后：① 编辑器提示「文件已在磁盘上更改」→ 点 **Revert**；② 点顶部 **Restart** 重启内核。

> 命令行的等价写法：`powershell -NoProfile -ExecutionPolicy Bypass -File tools\reset_practice.ps1`（加 `-NoBackup` 不备份）。